# Day 13 — Fashion MNIST Image Classifier using a CNN

Same dataset as Day 12, completely different architecture.

| | Day 12 — ANN | Day 13 — CNN |
|---|---|---|
| First thing it does to the image | `Flatten` it into 784 unrelated numbers | Slides small filters **over the picture** |
| Does it know pixel 5 is next to pixel 6? | No | Yes |
| Test accuracy | 86.31% | *(this notebook)* |

**The one-line reason CNNs win on images:** a CNN keeps the picture a picture.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist

SEED, EPOCHS, BATCH_SIZE, VALIDATION_SPLIT = 42, 15, 64, 0.2
keras.utils.set_random_seed(SEED)

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

## 1. Load the dataset

60,000 images to learn from, 10,000 sealed away to mark us at the end.

In [ ]:
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

print("Training images :", X_train.shape, "  labels:", y_train.shape)
print("Test images     :", X_test.shape, "  labels:", y_test.shape)
print("Pixel range     :", X_train.min(), "to", X_train.max(), f"(dtype {X_train.dtype})")

## 2. Look at the images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(f"{CLASS_NAMES[y_train[i]]}\n(label {y_train[i]})", fontsize=10)
    ax.axis("off")
fig.suptitle("Fashion MNIST — Sample Training Images", fontsize=16, fontweight="bold")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for label, ax in enumerate(axes.flat):
    ax.imshow(X_train[np.where(y_train == label)[0][0]], cmap="gray")
    ax.set_title(f"{label}: {CLASS_NAMES[label]}", fontsize=11)
    ax.axis("off")
fig.suptitle("One Example of Each Class", fontsize=16, fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Normalize, and add the channel dimension

**Normalize** — pixels arrive as 0–255. Divide by 255 so they land between 0 and 1. Big input numbers make gradient descent overshoot, and Keras initialises its weights assuming small inputs.

**The extra step vs Day 12** — a `Dense` layer wants a flat list, so the ANN never needed to know about colour. `Conv2D` works *on the picture*, so it must be told how many colour channels there are: `1` = greyscale, `3` = RGB.

`(60000, 28, 28)` → `(60000, 28, 28, 1)`

In [ ]:
X_train = np.expand_dims(X_train.astype("float32") / 255.0, -1)
X_test  = np.expand_dims(X_test.astype("float32") / 255.0, -1)

print("Training shape :", X_train.shape)
print("Test shape     :", X_test.shape)
print("Pixel range    :", X_train.min(), "to", X_train.max())

## 4. What a convolution actually does

A filter is just **9 numbers in a 3×3 square**. Slide it over the picture, multiply the 9 pixels underneath by the 9 filter numbers, add them up, write down the answer, move one pixel right.

Below the filters are hand-written so you can see the effect. In a real CNN nobody writes them — training discovers them.

In [ ]:
def convolve(img, kernel):
    k = kernel.shape[0]
    out = np.zeros((img.shape[0] - k + 1, img.shape[1] - k + 1), dtype="float32")
    for r in range(out.shape[0]):
        for c in range(out.shape[1]):
            out[r, c] = np.sum(img[r:r+k, c:c+k] * kernel)
    return out

filters = {
    "Vertical edges":   np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype="float32"),
    "Horizontal edges": np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype="float32"),
    "Blur":             np.ones((3, 3), dtype="float32") / 9.0,
}

img = X_test[0].squeeze()
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("Original", fontweight="bold"); axes[0].axis("off")
for ax, (name, k) in zip(axes[1:], filters.items()):
    ax.imshow(convolve(img, k), cmap="gray"); ax.set_title(name, fontweight="bold"); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Data augmentation — the concept

Free extra training data: show slightly changed copies of the images you already have. A shirt flipped left-to-right is still a shirt, so the model learns *shirt-ness* instead of memorising one exact arrangement of pixels.

Augmentation layers are **on during training and automatically off at prediction time**.

⚠️ Choose carefully — horizontal flip is fine for clothes, but a mirrored handwritten `2` is not a `2`.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
])

sample = X_train[7:8]
fig, axes = plt.subplots(1, 6, figsize=(16, 3.2))
axes[0].imshow(sample[0].squeeze(), cmap="gray")
axes[0].set_title(f"ORIGINAL\n{CLASS_NAMES[y_train[7]]}", fontweight="bold", fontsize=10)
axes[0].axis("off")
for ax in axes[1:]:
    ax.imshow(augment(sample, training=True)[0].numpy().squeeze(), cmap="gray")
    ax.set_title("augmented", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Build the CNN

```
Conv2D(32) → MaxPool → Conv2D(64) → MaxPool → Conv2D(64) → Flatten → Dense(128) → Dropout → Dense(10)
  find edges      shrink   find shapes    shrink   abstract      unroll    decide     anti-      answer
                                                                                    memorising
```

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1), name="input_layer"),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv_1"),
    layers.MaxPooling2D((2, 2), name="pool_1"),          # 28x28 -> 14x14

    layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv_2"),
    layers.MaxPooling2D((2, 2), name="pool_2"),          # 14x14 -> 7x7

    layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv_3"),

    layers.Flatten(name="flatten"),                      # 7*7*64 = 3136
    layers.Dense(128, activation="relu", name="dense_1"),
    layers.Dropout(0.3, name="dropout"),                 # training only
    layers.Dense(10, activation="softmax", name="output_layer"),
], name="fashion_mnist_cnn")

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

## 7. Train

- **Epoch** = one full pass over all the training images.
- **Batch size** = how many images it looks at before adjusting the weights. 48,000 ÷ 64 = 750 updates per epoch.
- **validation_split=0.2** = 12,000 images held back from training so we can watch for overfitting *without* touching the test set.

In [ ]:
history = model.fit(X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_split=VALIDATION_SPLIT,
                    verbose=2)

## 8. Evaluate on the sealed test set

In [ ]:
train_acc = history.history["accuracy"][-1]
val_acc   = history.history["val_accuracy"][-1]
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"Training accuracy   : {train_acc*100:.2f}%")
print(f"Validation accuracy : {val_acc*100:.2f}%")
print(f"Test accuracy       : {test_acc*100:.2f}%")
print(f"Test loss           : {test_loss:.4f}")
print(f"\nDay 12 ANN scored 86.31%. Difference: {test_acc*100 - 86.31:+.2f} points")

## 9. Training & validation curves

| Shape | Meaning |
|---|---|
| Both improving together | Healthy learning |
| Train improves, validation flat | Overfitting — memorising |
| Validation **loss** turning upward | Stop. It is getting worse on unseen data. |

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(epochs_range, history.history["accuracy"], "o-", color="#0ea5e9", label="Training accuracy")
ax1.plot(epochs_range, history.history["val_accuracy"], "s--", color="#f97316", label="Validation accuracy")
ax1.axhline(test_acc, color="#22c55e", ls=":", label=f"Test accuracy ({test_acc:.4f})")
ax1.set_title("Model Accuracy", fontweight="bold"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
ax1.legend(loc="lower right"); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history.history["loss"], "o-", color="#0ea5e9", label="Training loss")
ax2.plot(epochs_range, history.history["val_loss"], "s--", color="#f97316", label="Validation loss")
ax2.set_title("Model Loss", fontweight="bold"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.legend(loc="upper right"); ax2.grid(alpha=0.3)

fig.suptitle("Fashion MNIST CNN — Training History", fontsize=16, fontweight="bold")
plt.tight_layout(); plt.show()

## 10. Predictions — predicted vs actual

In [ ]:
predictions = model.predict(X_test, verbose=0)
predicted_labels = predictions.argmax(axis=1)
confidences = predictions.max(axis=1)

print("First test image, full probability breakdown:")
for i, prob in enumerate(predictions[0]):
    mark = "  <-- highest" if i == predicted_labels[0] else ""
    print(f"  {i} {CLASS_NAMES[i]:<15} {prob:.6f} {'#' * int(prob*40)}{mark}")
print(f"\nPredicted: {CLASS_NAMES[predicted_labels[0]]}   Actual: {CLASS_NAMES[y_test[0]]}")

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 10))
for i, ax in enumerate(axes.flat):
    pred, actual = predicted_labels[i], y_test[i]
    ok = pred == actual
    ax.imshow(X_test[i].squeeze(), cmap="gray")
    ax.set_title(f"Predicted: {CLASS_NAMES[pred]}\nActual: {CLASS_NAMES[actual]}\n"
                 f"Confidence: {confidences[i]:.1%}",
                 fontsize=9, fontweight="bold", color="#15803d" if ok else "#dc2626")
    ax.axis("off")
fig.suptitle("Sample Predictions  (green = correct, red = wrong)", fontsize=16, fontweight="bold")
plt.tight_layout(); plt.show()

## 11. Per-class accuracy and the confusion matrix

Read a **row**: “of all the real Shirts, where did they get sent?” The diagonal is correct; everything off it is a confusion.

In [ ]:
per_class = []
print(f"{'Class':<15} {'Accuracy'}")
print("-" * 26)
for label in range(10):
    mask = y_test == label
    acc = (predicted_labels[mask] == label).mean()
    per_class.append(acc)
    print(f"{CLASS_NAMES[label]:<15} {acc:.4f}")
print(f"\nEasiest : {CLASS_NAMES[int(np.argmax(per_class))]}")
print(f"Hardest : {CLASS_NAMES[int(np.argmin(per_class))]}")

In [ ]:
cm = np.zeros((10, 10), dtype=int)
for actual, pred in zip(y_test, predicted_labels):
    cm[actual, pred] += 1

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted label", fontweight="bold"); ax.set_ylabel("Actual label", fontweight="bold")
ax.set_title("Confusion Matrix — Fashion MNIST CNN", fontsize=14, fontweight="bold")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > cm.max()/2 else "black")
fig.colorbar(im, ax=ax, label="Number of images")
plt.tight_layout(); plt.show()

## 12. Ten right, ten wrong

In [ ]:
correct_idx = np.where(predicted_labels == y_test)[0]
wrong_idx   = np.where(predicted_labels != y_test)[0]
print(f"Correct: {len(correct_idx):,}   Wrong: {len(wrong_idx):,}")

for title, idxs, colour in [("10 CORRECTLY Classified", correct_idx[:10], "#15803d"),
                            ("10 INCORRECTLY Classified", wrong_idx[:10], "#dc2626")]:
    fig, axes = plt.subplots(2, 5, figsize=(15, 7))
    for ax, idx in zip(axes.flat, idxs):
        ax.imshow(X_test[idx].squeeze(), cmap="gray")
        ax.set_title(f"Predicted: {CLASS_NAMES[predicted_labels[idx]]}\n"
                     f"Actual: {CLASS_NAMES[y_test[idx]]}\n"
                     f"Confidence: {confidences[idx]:.1%}",
                     fontsize=9, color=colour, fontweight="bold")
        ax.axis("off")
    fig.suptitle(title, fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()

## 13. Look inside — the learned filters and the feature maps

The 32 filters below started as **random numbers**. Training turned them into pattern detectors. Nobody designed them.

In [ ]:
conv1_w = model.get_layer("conv_1").get_weights()[0]
fig, axes = plt.subplots(4, 8, figsize=(12, 6.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(conv1_w[:, :, 0, i], cmap="viridis"); ax.set_title(f"f{i}", fontsize=8); ax.axis("off")
fig.suptitle("The 32 learned 3×3 filters of conv_1", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
extractor = keras.Model(inputs=model.inputs,
                        outputs=[model.get_layer(n).output for n in ["conv_1", "pool_1", "conv_2", "pool_2"]])
maps = extractor.predict(X_test[:1], verbose=0)

for name, fmap in zip(["conv_1", "pool_1", "conv_2", "pool_2"], maps):
    fig, axes = plt.subplots(1, 8, figsize=(15, 2.2))
    for c, ax in enumerate(axes):
        ax.imshow(fmap[0, :, :, c], cmap="viridis"); ax.axis("off")
    fig.suptitle(f"{name} — shape {fmap.shape[1:]}", fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()

## 14. Save the model

In [ ]:
os.makedirs("outputs", exist_ok=True)
model.save("outputs/fashion_mnist_cnn.keras")
print("Saved: outputs/fashion_mnist_cnn.keras")

---

## What this notebook showed

1. **A CNN keeps the picture a picture.** The ANN's `Flatten` destroyed the fact that pixel 5 sits next to pixel 6. `Conv2D` never loses it.
2. **A filter is 9 numbers, reused everywhere.** That reuse is why a CNN needs far fewer parameters in its convolution layers than a Dense layer would, and why it can spot a sleeve wherever it appears.
3. **Pooling makes it forgiving.** Keeping the max of each 2×2 block means "the edge is roughly here" instead of "the edge is at pixel (13, 7)".
4. **The remaining mistakes are honest ones.** They cluster among T-shirt / Pullover / Coat / Shirt — garments that genuinely look alike at 28×28.